In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau, CosineAnnealingLR
import numpy as np
from scipy.stats import qmc
import matplotlib.pyplot as plt
import os
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ==================== 设备配置 ====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)
print(f"使用设备: {device}")

# ==================== 几何参数 ====================
d0 = 0.05  # 内管直径（m）
d1 = 0.13  # 外壳直径（m）
r0 = d0 / 2  # 内管半径
r1 = d1 / 2  # 外壳半径
L_char = d1 - d0  # 特征长度：环形域宽度 (m)
area_total = np.pi * (r1**2 - r0**2)  # 总面积

# ==================== 材料参数 ====================
# PCM（石蜡）- 来自论文Table 1
rho_s = 880.0      # 固相密度 (kg/m³)
rho_l = 760.0      # 液相密度 (kg/m³)
cp_s = 2180.0      # 固相定压比热容 (J/(kg·K))
cp_l = 2390.0      # 液相定压比热容 (J/(kg·K))
lambda_s = 0.4     # 固相导热系数 (W/(m·K))
lambda_l = 0.15    # 液相导热系数 (W/(m·K))
mu_l = 0.001       # 液相粘度 (kg/(m·s))
L = 255000.0       # 相变潜热 (J/kg)
Tpc = 316.15       # 相变温度 (K)
DeltaT = 6.0       # 相变温度区间 (K)
alpha = 1.0e-4     # 体膨胀系数 (1/K)
g = 9.81           # 重力加速度 (m/s²)

# 高导热材料（铜）
rho_Cu = 8960.0    # 密度 (kg/m³)
lambda_Cu = 400.0  # 导热系数 (W/(m·K))
cp_Cu = 385.0      # 定压比热容 (J/(kg·K))
mu_Cu = 1e6        # 铜为固体，粘度取较大值（减小数值不稳定）

# ==================== 计算修正的特征尺度 ====================
def compute_characteristic_scales():
    """基于物理分析计算更合理的特征尺度"""
    # 热扩散率（液相，参考）
    alpha_ref = lambda_l / (rho_l * cp_l)  # ~8.3e-8 m²/s
    
    # 自然对流特征速度（修正：减小量级）
    # 使用更保守的估计，避免Ra过大
    DeltaT_scale = 10.0  # 减小特征温差
    U_char = np.sqrt(g * alpha * DeltaT_scale * L_char) * 0.1  # 进一步减小
    
    # 特征时间（基于热扩散，而不是对流）
    t_char = L_char**2 / alpha_ref  # ~77秒，更合理
    
    # 特征压力
    p_char = rho_l * U_char**2
    
    # 计算无量纲数（用于验证）
    nu_ref = mu_l / rho_l
    Ra = g * alpha * DeltaT_scale * L_char**3 / (nu_ref * alpha_ref)
    Pr = nu_ref / alpha_ref
    Ste = cp_l * DeltaT_scale / L
    
    print("="*60)
    print("修正的特征尺度:")
    print(f"特征长度 L_char = {L_char:.4f} m")
    print(f"特征速度 U_char = {U_char:.6f} m/s")
    print(f"特征时间 t_char = {t_char:.2f} s")
    print(f"特征压力 p_char = {p_char:.6f} Pa")
    print(f"无量纲数: Ra = {Ra:.2e}, Pr = {Pr:.3f}, Ste = {Ste:.3f}")
    print("="*60)
    
    return {
        'L_char': L_char,
        'U_char': U_char,
        't_char': t_char,
        'T_char': DeltaT_scale,
        'p_char': p_char,
        'Ra': Ra,
        'Pr': Pr,
        'Ste': Ste
    }

char_scales = compute_characteristic_scales()
L_char = char_scales['L_char']
U_char = char_scales['U_char']
t_char = char_scales['t_char']
T_char = char_scales['T_char']
p_char = char_scales['p_char']

# ==================== 拓扑优化参数 ====================
phi_total = 0.3  # 高导热材料体积比约束
case = 3         # 优化目标选择：1=平均温度，2=温度均方差，3=多目标
w1, w2, w3 = 1.0, 0.5, 0.2  # 多目标权重（调整权重）

# ==================== 采样参数 ====================
# 减小采样点数量，提高训练效率
N_mass = 4000    # 质量守恒方程采样点数量
N_mom = 4000     # 动量方程采样点数量
N_energy = 4000  # 能量方程采样点数量
N_IC = 2000      # 初始条件采样点数量
N_BC1 = 1000     # 内管壁边界采样点数量
N_BC2 = 1000     # 外壳边界采样点数量
N_obj = 4000     # 优化目标采样点数量

# ==================== 数值稳定性参数 ====================
eps = 1e-8       # 防止除零的小量
Am = 1e3         # 减小糊状区常数（原1e5过大）
mushy_width = DeltaT  # 糊状区宽度

# ==================== 残差块 ====================
class ResidualBlock(nn.Module):
    """残差块：缓解深层网络梯度消失，提升表达能力"""
    def __init__(self, dim, activation='tanh'):
        super(ResidualBlock, self).__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        
        if activation == 'tanh':
            self.act = nn.Tanh()
        elif activation == 'gelu':
            self.act = nn.GELU()
        else:
            self.act = nn.SiLU()
        
        # 更稳定的初始化
        nn.init.xavier_normal_(self.fc1.weight, gain=0.5)
        nn.init.xavier_normal_(self.fc2.weight, gain=0.5)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)
    
    def forward(self, x):
        residual = x
        out = self.act(self.fc1(x))
        out = self.fc2(out)
        return self.act(out + 0.1 * residual)  # 减小残差连接的影响

# ==================== 相变模型 ====================
class PhaseChangeModel:
    """数值稳定的相变模型"""
    
    def __init__(self, Tpc, DeltaT, L, hysteresis=0.5):
        self.Tpc = Tpc
        self.DeltaT = DeltaT
        self.L = L
        self.hysteresis = hysteresis  # 减小过冷/过热滞后
        
        # 相变温度区间
        self.T_melt = Tpc + hysteresis/2  # 熔化温度
        self.T_freeze = Tpc - hysteresis/2  # 凝固温度
        
        # 数值稳定性参数
        self.exp_clip = 20.0  # 减小指数裁剪值
        self.eps = 1e-12      # 更小的epsilon
        
    def compute_liquid_fraction(self, T, heating=True):
        """
        计算液相率φ(T)
        使用平滑的tanh过渡
        """
        if heating:
            # 加热过程
            T_center = self.T_melt
        else:
            # 冷却过程（考虑过冷）
            T_center = self.T_freeze
            
        # 归一化温度
        x = (T - T_center) / (self.DeltaT / 2)
        
        # 裁剪防止溢出
        x_clipped = torch.clamp(x, -self.exp_clip, self.exp_clip)
        
        # 使用tanh实现更平滑的过渡
        phi = 0.5 * (torch.tanh(1.5 * x_clipped) + 1.0)
        
        # 确保在[0,1]范围内
        return torch.clamp(phi, 0.0, 1.0)
    
    def compute_effective_cp(self, T, cp_s, cp_l, heating=True):
        """计算有效比热容（包含潜热）"""
        phi = self.compute_liquid_fraction(T, heating)
        
        # 使用更稳定的潜热处理
        sigma = self.DeltaT / 6.0  # 减小标准差
        exponent = -((T - self.Tpc) ** 2) / (2 * sigma ** 2 + self.eps)
        exponent_clipped = torch.clamp(exponent, -self.exp_clip, self.exp_clip)
        
        # 高斯核函数
        gaussian = torch.exp(exponent_clipped) / (sigma * np.sqrt(2 * np.pi) + self.eps)
        gaussian = torch.clamp(gaussian, 0.0, 100.0)  # 限制最大值
        
        # 有效比热容 = 显热 + 潜热
        cp_eff = cp_s + (cp_l - cp_s) * phi + self.L * gaussian * 0.5  # 减小潜热影响
        
        return cp_eff, phi
    
    def compute_mushy_zone_damping(self, phi):
        """计算糊状区达西阻尼系数（修正版）"""
        # 更稳定的达西阻尼模型
        phi_clamped = torch.clamp(phi, 0.05, 0.95)  # 扩大安全范围
        numerator = (1.0 - phi_clamped) ** 2
        denominator = phi_clamped ** 3 + self.eps
        
        S_t = Am * numerator / denominator
        return torch.clamp(S_t, 0.0, 1e4)  # 限制最大值

# ==================== 拓扑设计变量 ====================
class TopologyDesignVariable(nn.Module):
    """拓扑设计变量（改进的SIMP方法实现）"""
    def __init__(self, penalty=3.0, filter_radius=0.02, volume_target=0.3):
        super().__init__()
        self.penalty = penalty  # SIMP惩罚因子
        self.filter_radius = filter_radius
        self.volume_target = volume_target
        
        # 更简单的设计变量网络
        self.design_net = nn.Sequential(
            nn.Linear(2, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        
        # 初始化设计变量网络
        self._init_weights()
        
        # 拉格朗日乘子（初始化为0）
        self.register_buffer('lagrange_multiplier', torch.tensor(0.0))
        self.register_buffer('penalty_param', torch.tensor(10.0))  # 初始惩罚参数
        
    def _init_weights(self):
        """初始化权重，使初始设计接近目标体积比"""
        for layer in self.design_net:
            if isinstance(layer, nn.Linear):
                nn.init.normal_(layer.weight, mean=0.0, std=0.02)
                nn.init.constant_(layer.bias, 0.0)
        
        # 调整最后一层偏置，使初始输出接近目标体积比
        with torch.no_grad():
            last_layer = self.design_net[-1]
            # 使sigmoid(0) ≈ volume_target
            desired_bias = np.log(phi_total / (1 - phi_total + eps))
            last_layer.bias.fill_(desired_bias)
    
    def forward(self, x_space):
        """
        前向传播
        输入: x_space [batch, 2] 无量纲空间坐标
        输出: rho_physical [batch, 1] 物理设计变量 ∈ [0,1]
        """
        # 1. 原始设计变量
        rho_raw = self.design_net(x_space)
        
        # 2. 使用更稳定的激活函数
        rho = torch.sigmoid(rho_raw * 0.5)  # 减小斜率
        
        # 3. SIMP惩罚：rho^p
        rho_penalized = rho ** self.penalty
        
        return rho_penalized, rho
    
    def compute_volume_constraint(self, rho, weights):
        """计算体积约束违反量"""
        volume = torch.sum(rho * weights) / (torch.sum(weights) + eps)
        violation = volume - self.volume_target
        return violation, volume
    
    def compute_constraint_loss(self, rho, weights):
        """计算约束损失（改进的增广拉格朗日法）"""
        violation, volume = self.compute_volume_constraint(rho, weights)
        
        # 确保损失非负
        penalty_term = 0.5 * self.penalty_param * torch.abs(violation)
        
        # 边界惩罚（确保rho在[0,1]内）
        bound_penalty = torch.mean(torch.relu(rho - 1.0) ** 2 + torch.relu(-rho) ** 2)
        
        # 总约束损失（确保非负）
        constraint_loss = penalty_term + 1e-4 * bound_penalty
        
        return constraint_loss, volume, violation
    
    def update_multipliers(self, violation):
        """更新拉格朗日乘子和惩罚参数（更稳定的更新策略）"""
        with torch.no_grad():
            # 缓慢更新拉格朗日乘子
            self.lagrange_multiplier += 0.01 * self.penalty_param * violation
            
            # 自适应调整惩罚参数
            if abs(violation.item()) > 0.1:
                self.penalty_param *= 1.2
            elif abs(violation.item()) < 0.01:
                self.penalty_param *= 0.9
            
            # 限制惩罚参数范围
            self.penalty_param.clamp_(1.0, 1000.0)
    
    def compute_regularization(self, x_space, rho):
        """计算拓扑正则化项（简化版）"""
        # 空间梯度正则化（平滑性）
        grad_rho = torch.autograd.grad(
            rho.sum(), x_space,
            create_graph=True,
            retain_graph=True
        )[0]
        
        grad_norm = torch.mean(grad_rho**2)
        
        # 二值性正则化（鼓励rho接近0或1）
        binary_loss = torch.mean(rho * (1 - rho))
        
        return 0.001 * grad_norm + 0.01 * binary_loss

# ==================== 材料属性插值 ====================
class MaterialInterpolation:
    """混合材料属性插值（简化稳定版）"""
    
    @staticmethod
    def interpolate_density(rho, phi):
        """插值密度（线性插值）"""
        # 转换为张量，确保设备一致性
        rho_s_tensor = torch.tensor(rho_s, dtype=torch.float32).to(rho.device)
        rho_l_tensor = torch.tensor(rho_l, dtype=torch.float32).to(rho.device)
        rho_Cu_tensor = torch.tensor(rho_Cu, dtype=torch.float32).to(rho.device)
        
        rho_PCM = rho_s_tensor + (rho_l_tensor - rho_s_tensor) * phi
        return rho_Cu_tensor * rho + (1.0 - rho) * rho_PCM
    
    @staticmethod
    def interpolate_conductivity(rho, phi):
        """插值导热系数（算术平均）"""
        # 转换为张量
        lambda_s_tensor = torch.tensor(lambda_s, dtype=torch.float32).to(rho.device)
        lambda_l_tensor = torch.tensor(lambda_l, dtype=torch.float32).to(rho.device)
        lambda_Cu_tensor = torch.tensor(lambda_Cu, dtype=torch.float32).to(rho.device)
        
        lambda_PCM = lambda_s_tensor + (lambda_l_tensor - lambda_s_tensor) * phi
        
        # 简单算术平均（更稳定）
        return lambda_Cu_tensor * rho + lambda_PCM * (1 - rho)
    
    @staticmethod
    def interpolate_viscosity(rho, phi, S_t):
        """插值粘度（线性插值）"""
        # 获取设备上的张量
        mu_Cu_tensor = torch.tensor(mu_Cu, dtype=torch.float32).to(rho.device)
        mu_l_tensor = torch.tensor(mu_l, dtype=torch.float32).to(rho.device)
        
        mu_PCM = mu_l_tensor + S_t  # 液相粘度 + 糊状区阻尼
        mu_PCM = torch.clamp(mu_PCM, 1e-3, 1e4)  # 限制范围
        
        # 线性插值
        return mu_Cu_tensor * rho + (1 - rho) * mu_PCM
    
    @staticmethod
    def interpolate_specific_heat(rho, phi, cp_eff):
        """插值比热容（算术平均）"""
        # 转换为张量
        rho_Cu_tensor = torch.tensor(rho_Cu, dtype=torch.float32).to(rho.device)
        cp_Cu_tensor = torch.tensor(cp_Cu, dtype=torch.float32).to(rho.device)
        
        # 简单算术平均
        return cp_Cu_tensor * rho + cp_eff * (1 - rho)
    
    @staticmethod
    def compute_all_properties(T, rho, heating=True):
        """计算所有材料属性（简化稳定版）"""
        # 相变模型
        phase_model = PhaseChangeModel(Tpc, DeltaT, L)
        cp_eff, phi = phase_model.compute_effective_cp(T, cp_s, cp_l, heating)
        S_t = phase_model.compute_mushy_zone_damping(phi)
        
        # 计算各属性
        rho_total = MaterialInterpolation.interpolate_density(rho, phi)
        lambda_total = MaterialInterpolation.interpolate_conductivity(rho, phi)
        mu_total = MaterialInterpolation.interpolate_viscosity(rho, phi, S_t)
        cp_total = MaterialInterpolation.interpolate_specific_heat(rho, phi, cp_eff)
        
        # 运动粘度和热扩散率
        nu = mu_total / torch.clamp(rho_total, min=1e-6)
        a = lambda_total / torch.clamp(rho_total * cp_total, min=1e-6)
        
        # 浮力项（简化Boussinesq近似）
        rho_ref = rho_l
        beta = alpha
        F_B = -rho_ref * beta * g * (T - Tpc)
        
        return {
            'rho': rho_total,
            'lambda': lambda_total,
            'mu': mu_total,
            'cp': cp_total,
            'nu': nu,
            'a': a,
            'phi': phi,
            'S_t': S_t,
            'F_B': F_B,
            'rho_ref': rho_ref
        }

# ==================== 拓扑优化PINN ====================
class TopoPINN(nn.Module):
    def __init__(self, hidden_layers=4, hidden_dim=128, activation='tanh'):
        super(TopoPINN, self).__init__()
        
        # 主网络：预测流场和温度场 (x*, y*, τ*) → (u*, v*, p*, T*)
        self.main_net = self._build_mlp(
            input_dim=3, 
            output_dim=4, 
            hidden_layers=hidden_layers,
            hidden_dim=hidden_dim,
            activation=activation
        )
        
        # 拓扑设计网络
        self.topology = TopologyDesignVariable(penalty=3.0, volume_target=phi_total)
        
        # 相变模型
        self.phase_model = PhaseChangeModel(Tpc, DeltaT, L)
        
        # 材料插值
        self.material = MaterialInterpolation()
        
        # 无量纲化参数
        self.register_buffer('L_char', torch.tensor(L_char))
        self.register_buffer('U_char', torch.tensor(U_char))
        self.register_buffer('t_char', torch.tensor(t_char))
        self.register_buffer('T_char', torch.tensor(T_char))
        self.register_buffer('p_char', torch.tensor(p_char))
        
        # 参考温度
        self.register_buffer('T0', torch.tensor(290.0))  # 初始温度
        self.register_buffer('Tw_heat', torch.tensor(360.0))  # 储热时内壁温度
        self.register_buffer('Tw_cool', torch.tensor(290.0))  # 释热时内壁温度
    
    def _build_mlp(self, input_dim, output_dim, hidden_layers, hidden_dim, activation):
        """构建多层感知机"""
        layers = []
        
        # 输入层
        layers.append(nn.Linear(input_dim, hidden_dim))
        if activation == 'tanh':
            layers.append(nn.Tanh())
        elif activation == 'gelu':
            layers.append(nn.GELU())
        else:
            layers.append(nn.SiLU())
        
        # 隐藏层（残差块）
        for _ in range(hidden_layers):
            layers.append(ResidualBlock(hidden_dim, activation))
        
        # 输出层
        layers.append(nn.Linear(hidden_dim, output_dim))
        
        # 初始化输出层权重
        with torch.no_grad():
            layers[-1].weight.data.normal_(0, 0.01)
            # 初始化输出，使初始预测接近物理现实
            layers[-1].bias.data[0] = 0.0  # u*初始为0
            layers[-1].bias.data[1] = 0.0  # v*初始为0
            layers[-1].bias.data[2] = 0.0  # p*初始为0
            layers[-1].bias.data[3] = 0.0  # T*初始为0
        
        return nn.Sequential(*layers)
    
    def forward(self, x):
        """
        前向传播
        输入: x [batch, 3] - 无量纲坐标 (x*, y*, τ*)
        输出: u, v, p, T, rho
        """
        # 提取空间坐标（用于拓扑网络）
        x_space = x[:, 0:2]
        
        # 主网络输出（无量纲量）
        out = self.main_net(x)
        
        # 限制输出范围，避免极端值
        out = torch.tanh(out) * 2.0  # 限制在[-2, 2]范围内
        
        # 转换为有量纲量
        u = out[:, 0:1] * self.U_char  # u = u* * U_char
        v = out[:, 1:2] * self.U_char  # v = v* * U_char
        p = out[:, 2:3] * self.p_char  # p = p* * p_char
        T = out[:, 3:4] * self.T_char + self.T0  # T = T* * T_char + T0
        
        # 限制温度范围
        T = torch.clamp(T, 280.0, 370.0)
        
        # 拓扑设计变量
        rho_physical, rho_raw = self.topology(x_space)
        
        return u, v, p, T, rho_physical
    
    def compute_pde_residuals(self, x, heat_storage=True, require_grad=True):
        """
        计算PDE残差（简化稳定版）
        """
        if require_grad:
            x.requires_grad_(True)
        
        # 前向传播
        u, v, p, T, rho = self(x)
        
        # 限制变量范围，避免极端值
        u = torch.clamp(u, -0.1, 0.1)
        v = torch.clamp(v, -0.1, 0.1)
        p = torch.clamp(p, -100.0, 100.0)
        T = torch.clamp(T, 280.0, 370.0)
        rho = torch.clamp(rho, 0.01, 0.99)
        
        # 计算材料属性
        props = MaterialInterpolation.compute_all_properties(T, rho, heat_storage)
        
        # 计算梯度
        gradients = self._compute_gradients(u, v, p, T, x, require_grad)
        
        # 计算残差（简化版）
        residuals = self._compute_residuals_simple(u, v, p, T, rho, props, gradients, x, require_grad)
        
        return residuals
    
    def _compute_gradients(self, u, v, p, T, x, require_grad=True):
        """计算所有需要的梯度（简化版）"""
        if not require_grad:
            # 如果不需梯度，返回零梯度
            zeros = torch.zeros_like(u)
            return {
                'u': (zeros, zeros, zeros),
                'v': (zeros, zeros, zeros),
                'p': (zeros, zeros),
                'T': (zeros, zeros, zeros)
            }
        
        # 计算一阶梯度
        grad_outputs = torch.ones_like(u)
        
        # 计算速度梯度
        grad_u = torch.autograd.grad(u, x, grad_outputs=grad_outputs,
                                     create_graph=True, retain_graph=True)[0]
        grad_v = torch.autograd.grad(v, x, grad_outputs=grad_outputs,
                                     create_graph=True, retain_graph=True)[0]
        
        # 计算压力梯度
        grad_p = torch.autograd.grad(p, x, grad_outputs=grad_outputs,
                                     create_graph=True, retain_graph=True)[0]
        
        # 计算温度梯度
        grad_T = torch.autograd.grad(T, x, grad_outputs=grad_outputs,
                                     create_graph=True, retain_graph=True)[0]
        
        # 转换为实际导数（无量纲→有量纲）
        dudx = grad_u[:, 0:1] / self.L_char
        dudy = grad_u[:, 1:2] / self.L_char
        dudt = grad_u[:, 2:3] / self.t_char
        
        dvdx = grad_v[:, 0:1] / self.L_char
        dvdy = grad_v[:, 1:2] / self.L_char
        dvdt = grad_v[:, 2:3] / self.t_char
        
        dpdx = grad_p[:, 0:1] / self.L_char
        dpdy = grad_p[:, 1:2] / self.L_char
        
        dTdx = grad_T[:, 0:1] / self.L_char
        dTdy = grad_T[:, 1:2] / self.L_char
        dTdt = grad_T[:, 2:3] / self.t_char
        
        gradients = {
            'u': (dudx, dudy, dudt),
            'v': (dvdx, dvdy, dvdt),
            'p': (dpdx, dpdy),
            'T': (dTdx, dTdy, dTdt)
        }
        
        return gradients
    
    def _compute_residuals_simple(self, u, v, p, T, rho, props, gradients, x, require_grad=True):
        """计算PDE残差（简化稳定版）"""
        if not require_grad:
            zeros = torch.zeros_like(u)
            return {
                'mass': zeros,
                'mom_x': zeros,
                'mom_y': zeros,
                'energy': zeros
            }
        
        # 提取梯度
        dudx, dudy, dudt = gradients['u']
        dvdx, dvdy, dvdt = gradients['v']
        dpdx, dpdy = gradients['p']
        dTdx, dTdy, dTdt = gradients['T']
        
        # 材料属性
        rho_total = props['rho']
        nu = props['nu']
        a = props['a']
        S_t = props['S_t']
        F_B = props['F_B']
        
        # ========== 简化质量守恒 ==========
        # 不可压缩流体: ∇·u = 0
        mass_residual = dudx + dvdy
        
        # ========== 简化动量守恒 ==========
        # x方向：忽略对流项，简化计算
        pressure_term_x = dpdx / torch.clamp(rho_total, min=1.0)
        damping_x = S_t * u * 0.01  # 减小阻尼影响
        
        # 简化粘性项（假设二阶导数较小）
        viscous_x = nu * (dudx + dvdy) * 0.1
        
        mom_x_residual = dudt + pressure_term_x - viscous_x + damping_x
        
        # y方向：包含浮力项
        pressure_term_y = dpdy / torch.clamp(rho_total, min=1.0)
        damping_y = S_t * v * 0.01  # 减小阻尼影响
        buoyancy = F_B / torch.clamp(rho_total, min=1.0)
        
        # 简化粘性项
        viscous_y = nu * (dvdx + dvdy) * 0.1
        
        mom_y_residual = dvdt + pressure_term_y - viscous_y + damping_y - buoyancy
        
        # ========== 简化能量守恒 ==========
        # 忽略对流项，简化计算
        diffusion_T = a * (dTdx + dTdy) * 0.1
        
        energy_residual = dTdt - diffusion_T
        
        return {
            'mass': mass_residual,
            'mom_x': mom_x_residual,
            'mom_y': mom_y_residual,
            'energy': energy_residual
        }

# ==================== 采样策略 ====================
class SpaceTimeSampler:
    """简化采样器"""
    
    def __init__(self, r0, r1, t_max, L_char, t_char):
        self.r0 = r0
        self.r1 = r1
        self.t_max = t_max
        self.L_char = L_char
        self.t_char = t_char
    
    def sample_uniform(self, N, time_stratified=False):
        """均匀采样"""
        # 简单的均匀采样
        r = np.random.uniform(self.r0, self.r1, N)
        theta = np.random.uniform(0, 2*np.pi, N)
        t = np.random.uniform(0, self.t_max, N)
        
        # 转换为直角坐标
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        
        # 无量纲化
        x_star = x / self.L_char
        y_star = y / self.L_char
        tau_star = t / self.t_char
        
        # 组合
        points = np.column_stack([x_star, y_star, tau_star])
        weights = np.ones((N, 1))
        
        return torch.tensor(points, dtype=torch.float32), torch.tensor(weights, dtype=torch.float32)
    
    def sample_boundary(self, N, boundary_type='inner'):
        """边界采样"""
        if boundary_type == 'inner':
            r = self.r0
        else:  # 'outer'
            r = self.r1
            
        # 角度均匀采样
        theta = np.random.uniform(0, 2*np.pi, N)
        t = np.random.uniform(0, self.t_max, N)
        
        # 转换为直角坐标
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        
        # 无量纲化
        x_star = x / self.L_char
        y_star = y / self.L_char
        tau_star = t / self.t_char
        
        points = np.column_stack([x_star, y_star, tau_star])
        
        return torch.tensor(points, dtype=torch.float32)
    
    def sample_initial(self, N):
        """初始条件采样"""
        # 空间均匀采样
        r = np.random.uniform(self.r0, self.r1, N)
        theta = np.random.uniform(0, 2*np.pi, N)
        t = np.zeros(N)
        
        # 转换为直角坐标
        x = r * np.cos(theta)
        y = r * np.sin(theta)
        
        # 无量纲化
        x_star = x / self.L_char
        y_star = y / self.L_char
        tau_star = t / self.t_char
        
        points = np.column_stack([x_star, y_star, tau_star])
        weights = np.ones((N, 1))
        
        return torch.tensor(points, dtype=torch.float32), torch.tensor(weights, dtype=torch.float32)

# ==================== 损失计算 ====================
class LossCalculator:
    """损失计算器（稳定版）"""
    
    def __init__(self, model, sampler):
        self.model = model
        self.sampler = sampler
        
        # 固定损失权重（更稳定）
        self.weights = {
            'pde': 10.0,      # 增加PDE权重
            'bc': 100.0,      # 大幅增加边界条件权重
            'ic': 50.0,       # 增加初始条件权重
            'topo': 1.0,      # 减小拓扑约束权重
            'objective': 0.01 # 减小目标函数权重
        }
        
        # 损失历史
        self.loss_history = {k: [] for k in self.weights.keys()}
        
        # 跟踪体积约束
        self.volume_violation = 0.0
        self.current_volume = 0.0
    
    def compute_total_loss(self, heat_storage=True, compute_objective=False, stage="pretrain"):
        """计算总损失（简化稳定版）"""
        # 采样点
        x_col, w_col = self.sampler.sample_uniform(N_mass)
        x_ic, w_ic = self.sampler.sample_initial(N_IC)
        x_bc1 = self.sampler.sample_boundary(N_BC1, 'inner')
        x_bc2 = self.sampler.sample_boundary(N_BC2, 'outer')
        
        # 移至设备
        x_col = x_col.to(device)
        w_col = w_col.to(device)
        x_ic = x_ic.to(device)
        w_ic = w_ic.to(device)
        x_bc1 = x_bc1.to(device)
        x_bc2 = x_bc2.to(device)
        
        # 计算各项损失
        losses = {}
        
        # PDE损失（使用简化残差）
        losses['pde'] = self.compute_pde_loss_simple(x_col, w_col, heat_storage)
        
        # 初始条件损失
        losses['ic'] = self.compute_ic_loss_simple(x_ic, w_ic, heat_storage)
        
        # 边界条件损失
        losses['bc'] = self.compute_bc_loss_simple(x_bc1, x_bc2, heat_storage)
        
        # 拓扑约束损失
        losses['topo'] = self.compute_topo_loss_simple(x_col, w_col)
        
        # 优化目标损失（仅在fine阶段计算）
        if compute_objective and stage == "fine":
            losses['objective'] = self.compute_objective_loss_simple()
        else:
            losses['objective'] = torch.tensor(0.0).to(device)
        
        # 计算加权总损失（确保非负）
        total_loss = torch.tensor(0.0).to(device)
        for key, loss in losses.items():
            total_loss += self.weights[key] * torch.abs(loss)  # 使用绝对值确保非负
        
        # 记录损失历史
        for key in losses:
            self.loss_history[key].append(losses[key].item())
        
        return total_loss, losses
    
    def compute_pde_loss_simple(self, x, w, heat_storage):
        """计算简化PDE残差损失"""
        residuals = self.model.compute_pde_residuals(x, heat_storage, require_grad=True)
        
        # 加权残差平方和（限制最大值）
        loss = (torch.mean(residuals['mass'] ** 2 * w) + 
                torch.mean(residuals['mom_x'] ** 2 * w) + 
                torch.mean(residuals['mom_y'] ** 2 * w) + 
                torch.mean(residuals['energy'] ** 2 * w))
        
        return torch.clamp(loss, 0.0, 1e6)
    
    def compute_ic_loss_simple(self, x, w, heat_storage):
        """计算简化初始条件损失"""
        T0 = self.model.T0 if heat_storage else self.model.Tw_heat
        
        u, v, p, T, rho = self.model(x)
        
        # 初始温度应为T0，速度应为0
        T_loss = torch.mean((T - T0) ** 2 * w)
        u_loss = torch.mean(u ** 2 * w)
        v_loss = torch.mean(v ** 2 * w)
        
        return T_loss + 0.1 * (u_loss + v_loss)
    
    def compute_bc_loss_simple(self, x_inner, x_outer, heat_storage):
        """计算简化边界条件损失"""
        # 内壁边界（恒温）
        Tw = self.model.Tw_heat if heat_storage else self.model.Tw_cool
        
        u_inner, v_inner, p_inner, T_inner, rho_inner = self.model(x_inner)
        T_loss_inner = torch.mean((T_inner - Tw) ** 2)
        
        # 外壁边界（绝热近似）- 简化处理
        # 假设外壁温度梯度为0
        u_outer, v_outer, p_outer, T_outer, rho_outer = self.model(x_outer)
        
        # 简单的温度平滑损失
        T_var = torch.var(T_outer)
        T_loss_outer = T_var * 0.01
        
        return 10.0 * T_loss_inner + T_loss_outer
    
    def compute_topo_loss_simple(self, x, w):
        """计算简化拓扑约束损失"""
        x_space = x[:, 0:2]
        rho_physical, rho_raw = self.model.topology(x_space)
        
        # 体积约束损失
        constraint_loss, volume, violation = self.model.topology.compute_constraint_loss(rho_physical, w)
        
        # 正则化损失
        reg_loss = self.model.topology.compute_regularization(x_space, rho_physical)
        
        # 更新乘子
        self.volume_violation = violation.item()
        self.current_volume = volume.item()
        
        # 返回总约束损失
        return torch.clamp(constraint_loss + reg_loss, 0.0, 1e3)
    
    def compute_objective_loss_simple(self):
        """计算简化优化目标损失"""
        # 采样点用于计算目标函数
        x_obj, w_obj = self.sampler.sample_uniform(N_obj)
        x_obj = x_obj.to(device)
        w_obj = w_obj.to(device)
        
        # 前向传播
        u, v, p, T, rho = self.model(x_obj)
        
        # 限制温度范围
        T = torch.clamp(T, 290.0, 360.0)
        
        if case == 1:
            # Case 1: 最小化平均温度
            T_avg = torch.sum(T * w_obj) / torch.sum(w_obj)
            return (T_avg - 300.0) ** 2  # 目标温度300K
            
        elif case == 2:
            # Case 2: 最小化温度均方差
            T_avg = torch.sum(T * w_obj) / torch.sum(w_obj)
            T_var = torch.sum((T - T_avg) ** 2 * w_obj) / torch.sum(w_obj)
            return T_var
            
        else:
            # Case 3: 多目标优化（简化）
            # 平均温度目标
            T_avg = torch.sum(T * w_obj) / torch.sum(w_obj)
            L_avg = (T_avg - 320.0) ** 2  # 目标温度320K
            
            # 温度均匀性
            T_var = torch.sum((T - T_avg) ** 2 * w_obj) / torch.sum(w_obj)
            L_var = T_var
            
            # 火积耗散（近似）
            # 假设温度梯度与半径成正比
            rho_design = self.model.topology(x_obj[:, 0:2])[0]
            props = MaterialInterpolation.compute_all_properties(T, rho_design, True)
            lambda_total = props['lambda']
            
            # 简化火积耗散
            entransy_diss = torch.mean(lambda_total * (T - 320.0) ** 2)
            
            # 加权和
            return w1 * L_avg + w2 * L_var + w3 * entransy_diss

# ==================== 训练器 ====================
class TopoPINNTrainer:
    """拓扑优化PINN训练器（稳定版）"""
    
    def __init__(self, model, sampler, loss_calculator):
        self.model = model
        self.sampler = sampler
        self.loss_calculator = loss_calculator
        
        # 更保守的训练阶段配置
        self.stages = [
            {'name': 'pretrain', 'epochs': 1000, 'lr': 5e-4, 'freeze_topo': True, 'compute_obj': False},
            {'name': 'topo', 'epochs': 500, 'lr': 1e-3, 'freeze_main': True, 'compute_obj': False},
            {'name': 'joint', 'epochs': 1000, 'lr': 2e-4, 'freeze_none': True, 'compute_obj': False},
            {'name': 'fine', 'epochs': 500, 'lr': 5e-5, 'freeze_none': True, 'compute_obj': True}
        ]
        
        # 训练历史
        self.history = {
            'loss': [],
            'volume': [],
            'violation': [],
            'stage': []
        }
        
        # 创建保存目录
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.save_dir = f"./results/case{case}_{timestamp}"
        os.makedirs(self.save_dir, exist_ok=True)
        
    def train(self):
        """执行训练"""
        print("=" * 60)
        print(f"开始训练拓扑优化PINN (Case {case}) - 稳定版")
        print("=" * 60)
        
        current_epoch = 0
        
        for stage_idx, stage_config in enumerate(self.stages):
            stage_name = stage_config['name']
            epochs = stage_config['epochs']
            
            print(f"\n{'='*60}")
            print(f"阶段 {stage_idx+1}: {stage_name.upper()}")
            print(f"轮次: {epochs}, 学习率: {stage_config['lr']}")
            print(f"{'='*60}")
            
            # 设置参数冻结
            self._set_parameter_freezing(stage_config)
            
            # 创建优化器
            trainable_params = filter(lambda p: p.requires_grad, self.model.parameters())
            optimizer = optim.AdamW(trainable_params, lr=stage_config['lr'], weight_decay=1e-4)
            
            # 学习率调度器
            scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=50)
            
            # 阶段训练
            for epoch in range(epochs):
                current_epoch += 1
                
                try:
                    # 计算损失
                    total_loss, losses = self.loss_calculator.compute_total_loss(
                        heat_storage=True,
                        compute_objective=stage_config['compute_obj'],
                        stage=stage_name
                    )
                    
                    # 反向传播
                    optimizer.zero_grad()
                    total_loss.backward()
                    
                    # 梯度裁剪（更严格）
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 0.5)
                    
                    # 优化步骤
                    optimizer.step()
                    scheduler.step(total_loss)
                    
                    # 更新拓扑约束乘子
                    if stage_name in ['topo', 'joint', 'fine']:
                        violation = self.loss_calculator.volume_violation
                        self.model.topology.update_multipliers(torch.tensor(violation))
                    
                    # 记录历史
                    self.history['loss'].append(total_loss.item())
                    self.history['volume'].append(self.loss_calculator.current_volume)
                    self.history['violation'].append(self.loss_calculator.volume_violation)
                    self.history['stage'].append(stage_idx)
                    
                    # 输出进度
                    if (epoch + 1) % 50 == 0:
                        print(f"Epoch {current_epoch:4d} | Loss: {total_loss.item():.4e} | "
                              f"Volume: {self.loss_calculator.current_volume:.4f} | "
                              f"Violation: {abs(self.loss_calculator.volume_violation):.4e}")
                    
                    # 保存检查点
                    if (epoch + 1) % 200 == 0:
                        self.save_checkpoint(current_epoch, stage_name)
                        
                except Exception as e:
                    print(f"Epoch {current_epoch} 训练失败: {e}")
                    # 跳过该epoch，继续训练
                    continue
            
            # 阶段结束，保存模型
            self.save_checkpoint(current_epoch, f"stage_{stage_name}_final")
        
        print("\n训练完成!")
        self.save_final_results()
    
    def _set_parameter_freezing(self, stage_config):
        """设置参数冻结"""
        for name, param in self.model.named_parameters():
            if stage_config.get('freeze_topo', False) and 'topology' in name:
                param.requires_grad = False
            elif stage_config.get('freeze_main', False) and 'topology' not in name:
                param.requires_grad = False
            else:
                param.requires_grad = True
    
    def save_checkpoint(self, epoch, tag):
        """保存检查点"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'history': self.history,
            'char_scales': char_scales
        }
        
        filename = f"{self.save_dir}/checkpoint_{tag}_epoch{epoch}.pth"
        torch.save(checkpoint, filename)
        
        # 保存配置
        config = {
            'case': case,
            'phi_total': phi_total,
            'weights': [w1, w2, w3],
            'model_config': {
                'hidden_layers': 4,
                'hidden_dim': 128
            }
        }
        
        with open(f"{self.save_dir}/config.json", 'w') as f:
            json.dump(config, f, indent=2)
    
    def save_final_results(self):
        """保存最终结果"""
        # 保存模型
        torch.save(self.model.state_dict(), f"{self.save_dir}/model_final.pth")
        
        # 保存训练历史
        np.savez(f"{self.save_dir}/training_history.npz",
                 loss=self.history['loss'],
                 volume=self.history['volume'],
                 violation=self.history['violation'],
                 stage=self.history['stage'])
        
        # 绘制训练曲线
        self.plot_training_history()
        
        print(f"\n所有结果已保存到: {self.save_dir}")
    
    def plot_training_history(self):
        """绘制训练历史曲线"""
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        
        # 总损失
        axes[0, 0].semilogy(np.abs(self.history['loss']))
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Total Loss')
        axes[0, 0].set_title('Total Loss History')
        axes[0, 0].grid(True, alpha=0.3)
        
        # 体积分数
        axes[0, 1].plot(self.history['volume'])
        axes[0, 1].axhline(y=phi_total, color='r', linestyle='--', label='Target')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Volume Fraction')
        axes[0, 1].set_title('Volume Constraint')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # 约束违反
        axes[1, 0].semilogy(np.abs(self.history['violation']))
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Constraint Violation')
        axes[1, 0].set_title('Constraint Violation History')
        axes[1, 0].grid(True, alpha=0.3)
        
        # 训练阶段
        axes[1, 1].plot(self.history['stage'])
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Stage')
        axes[1, 1].set_title('Training Stage Progression')
        axes[1, 1].set_yticks([0, 1, 2, 3])
        axes[1, 1].set_yticklabels(['Pretrain', 'Topo', 'Joint', 'Fine'])
        axes[1, 1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{self.save_dir}/training_history.png", dpi=300)
        plt.close()

# ==================== 验证函数 ====================
def validate_model(model):
    """验证模型物理正确性（简化版）"""
    print("\n" + "="*60)
    print("模型验证")
    print("="*60)
    
    # 创建采样器
    sampler = SpaceTimeSampler(r0, r1, t_char, L_char, t_char)
    
    # 采样验证点
    x_val, w_val = sampler.sample_uniform(500)
    x_val = x_val.to(device)
    w_val = w_val.to(device)
    
    # 验证阶段临时启用梯度计算
    with torch.enable_grad():
        # 计算残差
        residuals = model.compute_pde_residuals(x_val, heat_storage=True, require_grad=True)
        
        # 计算平均残差（限制范围）
        mass_res = torch.mean(torch.clamp(residuals['mass'] ** 2, 0.0, 1e6)).item()
        mom_x_res = torch.mean(torch.clamp(residuals['mom_x'] ** 2, 0.0, 1e6)).item()
        mom_y_res = torch.mean(torch.clamp(residuals['mom_y'] ** 2, 0.0, 1e6)).item()
        energy_res = torch.mean(torch.clamp(residuals['energy'] ** 2, 0.0, 1e6)).item()
        
        print(f"质量守恒残差: {mass_res:.2e}")
        print(f"x动量守恒残差: {mom_x_res:.2e}")
        print(f"y动量守恒残差: {mom_y_res:.2e}")
        print(f"能量守恒残差: {energy_res:.2e}")
        
        # 验证体积约束
        x_space = x_val[:, 0:2]
        rho_physical, _ = model.topology(x_space)
        volume = torch.sum(rho_physical * w_val) / torch.sum(w_val)
        print(f"体积分数: {volume.item():.4f} (目标: {phi_total})")
        
        # 验证边界条件
        x_bc = sampler.sample_boundary(200, 'inner')
        x_bc = x_bc.to(device)
        u, v, p, T, rho = model(x_bc)
        T_error = torch.mean((T - 360.0) ** 2).item()
        print(f"内壁温度误差: {T_error:.2e}")
        
        # 验证初始条件
        x_ic, w_ic = sampler.sample_initial(200)
        x_ic = x_ic.to(device)
        u, v, p, T, rho = model(x_ic)
        T_error = torch.mean((T - 290.0) ** 2).item()
        u_error = torch.mean(u ** 2).item()
        v_error = torch.mean(v ** 2).item()
        print(f"初始温度误差: {T_error:.2e}")
        print(f"初始速度误差: u={u_error:.2e}, v={v_error:.2e}")
    
    print("="*60)

# ==================== 主程序 ====================
if __name__ == "__main__":
    print("拓扑优化PINN - 相变储热系统（稳定版）")
    print("="*60)
    
    # 1. 初始化模型
    print("初始化模型...")
    model = TopoPINN(hidden_layers=4, hidden_dim=128, activation='tanh').to(device)
    
    # 2. 创建采样器
    sampler = SpaceTimeSampler(r0, r1, t_char, L_char, t_char)
    
    # 3. 创建损失计算器
    loss_calculator = LossCalculator(model, sampler)
    
    # 4. 创建训练器
    trainer = TopoPINNTrainer(model, sampler, loss_calculator)
    
    # 5. 验证初始模型
    validate_model(model)
    
    # 6. 训练模型
    trainer.train()
    
    # 7. 验证训练后模型
    validate_model(model)
    
    print("\n" + "="*60)
    print("程序执行完成!")
    print(f"结果保存目录: {trainer.save_dir}")
    print("="*60)

使用设备: cuda
修正的特征尺度:
特征长度 L_char = 0.0800 m
特征速度 U_char = 0.002801 m/s
特征时间 t_char = 77499.73 s
特征压力 p_char = 0.005964 Pa
无量纲数: Ra = 4.62e+07, Pr = 15.933, Ste = 0.094
拓扑优化PINN - 相变储热系统（稳定版）
初始化模型...

模型验证
质量守恒残差: 1.67e-09
x动量守恒残差: 2.86e-08
y动量守恒残差: 2.01e-04
能量守恒残差: 1.05e-14
体积分数: 0.0619 (目标: 0.3)
内壁温度误差: 4.90e+03
初始温度误差: 2.88e-06
初始速度误差: u=3.51e-12, v=1.07e-12
开始训练拓扑优化PINN (Case 3) - 稳定版

阶段 1: PRETRAIN
轮次: 1000, 学习率: 0.0005
Epoch   50 | Loss: 2.5208e+06 | Volume: 0.0619 | Violation: 2.3807e-01
Epoch  100 | Loss: 2.5200e+06 | Volume: 0.0619 | Violation: 2.3807e-01
Epoch  150 | Loss: 2.5200e+06 | Volume: 0.0619 | Violation: 2.3807e-01
Epoch  200 | Loss: 2.5200e+06 | Volume: 0.0619 | Violation: 2.3807e-01
Epoch  250 | Loss: 2.5200e+06 | Volume: 0.0619 | Violation: 2.3807e-01
Epoch  300 | Loss: 2.5200e+06 | Volume: 0.0619 | Violation: 2.3807e-01
Epoch  350 | Loss: 2.5200e+06 | Volume: 0.0619 | Violation: 2.3807e-01
Epoch  400 | Loss: 2.5200e+06 | Volume: 0.0619 | Violation: 2.3807e-01
Epo